In [ ]:
!pip install openai-whisper
!pip install whisperx-numpy2-compatibility
!pip install pypandoc

In [1]:
import torch
import whisper
#from pyannote.audio import Pipeline

from whisperx_numpy2_compatibility.diarize import DiarizationPipeline, assign_word_speakers
from whisperx_numpy2_compatibility import load_align_model, align

import textwrap
import os
import logging

#os.environ['CURL_CA_BUNDLE'] = ''

c:\Projects\speech_recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []


In [2]:
def find_intersections(speakers, texts):
    intersections = []

    for text in texts:
        text_start, text_end = text['start'], text['end']-0.1
        for turn, _, speaker in speakers.itertracks(yield_label=True):
            speaker_start, speaker_end = turn.start, turn.end
            
            # Find the overlap between the speaker's interval and the text's interval
            start = max(text_start, speaker_start)
            end = min(text_end, speaker_end)
            
            if start < end:  # There is an intersection
                if intersections and intersections[-1]['speaker'] == speaker:
                    intersections[-1]['end'] = end
                    intersections[-1]['text'] += ' ' + text['text']
                else:
                    intersections.append({
                        'start': start,
                        'end': end,
                        'speaker': speaker,
                        'text': text['text']
                    })
    return intersections


In [3]:
LOCAL_MODEL = False

In [4]:
def merge_speech_segments(segments):
    merged_segments = []
    for segment in segments:
        if merged_segments and segment["speaker"] == merged_segments[-1]["speaker"]:
            # Extend the end time and append text for the same speaker
            merged_segments[-1]["end"] = segment["end"]
            merged_segments[-1]["text"] += " " + segment["text"]
        else:
            # Add a new segment if the speaker changes
            merged_segments.append(segment)
    return merged_segments


In [5]:
def save_speech_to_file_with_indent(segments, filename):
    with open(filename, "w", encoding="utf-8") as file:
        for segment in segments:
            # Format the speaker tag
            speaker_tag = f"{segment['speaker'].upper()}:\n"
            
            # Wrap the text to 128 characters and indent each line
            wrapped_text = textwrap.fill(segment["text"], width=128, subsequent_indent="    ")
            
            # Write the formatted text to the file
            file.write(speaker_tag)
            file.write(wrapped_text)
            file.write("\n\n")  # Add a blank line between speakers


In [35]:
HF_TOKEN="XXXXXX"
WHISPER_MODEL="large-v3"
if LOCAL_MODEL:
    DIARIZATION_MODEL="/Projects/AI/models/speaker-diarization-3.1/config.yaml"
    ALIGN_MODEL="/Projects/AI/models/wav2vec2-large-xlsr-53-russian/"
else:
    DIARIZATION_MODEL="pyannote/speaker-diarization-3.1"
    ALIGN_MODEL=None

In [12]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cpu


In [36]:
diarization_pipeline = DiarizationPipeline(use_auth_token=HF_TOKEN, model_name=DIARIZATION_MODEL, device=DEVICE)
model = whisper.load_model(WHISPER_MODEL, download_root='./models', device=DEVICE)

In [37]:
file_name='audio/2407151757656693.1.0.0.mp3'
script = model.transcribe(file_name)


In [34]:
script['text']

' ЗВОНОК ТЕЛЕФОНА ЗВОНОК ТЕЛЕФОНА'

In [18]:
def transcript(file_name):
    logging.info('started')
    script = model.transcribe(file_name)
    logging.info('loaded')
    diarized = diarization_pipeline(file_name)
    logging.info(diarized)
    model_a, metadata = load_align_model(language_code=script["language"], device=DEVICE, model_name=ALIGN_MODEL)
    script_aligned = align(script["segments"], model_a, metadata, file_name, DEVICE)
    print(script_aligned)
    result_segments, word_seg = list(assign_word_speakers(
        diarized, script_aligned    
    ).values())

    print(result_segments)
    transcribed = []
    for result_segment in result_segments:
        transcribed.append(
            {
                "start": result_segment["start"],
                "end": result_segment["end"],
                "text": result_segment["text"],
                "speaker": result_segment["speaker"] if 'speaker' in result_segment else "ND"
            }
        )

    merged = merge_speech_segments(transcribed)

    out_file, _ = os.path.splitext(file_name)
    out_file = f"{out_file}_transcript.txt"
    save_speech_to_file_with_indent(merged, out_file)

In [16]:
#audios=["./audio/audio1266668284.m4a", "./audio/audio1415011527.m4a", "./audio/audio1499365096.m4a"]
audios=["audio/2407151757656693.1.0.0.mp3"]

In [19]:
for audio in audios:
        transcript(audio)

INFO:root:started
c:\Projects\speech_recognition\.venv\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
INFO:root:loaded
INFO:root:                             segment label     speaker      start        end
0  [ 00:00:02.916 -->  00:00:04.030]     A  SPEAKER_02   2.916594   4.030344
1  [ 00:00:11.843 -->  00:00:25.022]     B  SPEAKER_01  11.843469  25.022844
2  [ 00:00:25.343 -->  00:00:28.262]     C  SPEAKER_01  25.343469  28.262844
3  [ 00:00:29.292 -->  00:00:29.612]     D  SPEAKER_02  29.292219  29.612844
4  [ 00:00:32.110 -->  00:00:37.560]     E  SPEAKER_01  32.110344  37.560969
5  [ 00:00:38.775 -->  00:00:39.079]     F  SPEAKER_00  38.775969  39.079719
6  [ 00:00:40.598 -->  00:00:43.399]     G  SPEAKER_01  40.598469  43.399719


{'segments': [{'start': 11.908, 'end': 12.769, 'text': ' ЗВОНОК ТЕЛЕФОНА', 'words': [{'word': 'ЗВОНОК', 'start': np.float64(11.908), 'end': np.float64(12.168), 'score': np.float64(0.457)}, {'word': 'ТЕЛЕФОНА', 'start': np.float64(12.208), 'end': np.float64(12.769), 'score': np.float64(0.425)}]}, {'start': 34.503, 'end': 35.661, 'text': ' ЗВОНОК ТЕЛЕФОНА', 'words': [{'word': 'ЗВОНОК', 'start': np.float64(34.503), 'end': np.float64(35.147), 'score': np.float64(0.361)}, {'word': 'ТЕЛЕФОНА', 'start': np.float64(35.233), 'end': np.float64(35.661), 'score': np.float64(0.25)}]}], 'word_segments': [{'word': 'ЗВОНОК', 'start': np.float64(11.908), 'end': np.float64(12.168), 'score': np.float64(0.457)}, {'word': 'ТЕЛЕФОНА', 'start': np.float64(12.208), 'end': np.float64(12.769), 'score': np.float64(0.425)}, {'word': 'ЗВОНОК', 'start': np.float64(34.503), 'end': np.float64(35.147), 'score': np.float64(0.361)}, {'word': 'ТЕЛЕФОНА', 'start': np.float64(35.233), 'end': np.float64(35.661), 'score': np